# ESCREVER INTRO AQUI

In [1]:
from types import SimpleNamespace
from cipher_base import BLOCK_SIZE
from ctr import random_nonce, apply_ctr
from feistel import Feistel80
from spn import SPN80

BLOCK_BYTES = BLOCK_SIZE // 8

In [2]:
MASTER_KEY = 0x4a8f3c11e792d504b86a
CIPHERS = {
	"S07": {
		"SBOX": [15 - x for x in range(16)],
		"SPN_PBOX": [(29 * i + 7) % 80 for i in range(80)],
		"FEISTEL_PBOX": [(17 * i + 9) % 40 for i in range(40)],
		"MASTER_KEY": MASTER_KEY
	},
	"F01": {
		"SBOX": [(x*x + 3) % 16 for x in range(16)],
		"SPN_PBOX": [i ^ 1 for i in range(80)],
		"FEISTEL_PBOX": [i ^ 1 for i in range(40)],
		"MASTER_KEY": MASTER_KEY
	},
	"F07": {
		"SBOX": [15 - x for x in range(16)],
		"SPN_PBOX": [(29 * i + 7) % 80 for i in range(80)],
		"FEISTEL_PBOX": [(17 * i + 9) % 40 for i in range(40)],
		"MASTER_KEY": MASTER_KEY
	}
}

ANALYSIS = {
  "S07": {
		"AVALANCHE": 0,
		"RANDOM_2.1": 0,
		"RANDOM_2.3": 0,
		"RANDOM_2.13": 0				
	},
	"F01": {
		"AVALANCHE": 0,
		"RANDOM_2.1": 0,
		"RANDOM_2.3": 0,
		"RANDOM_2.13": 0				
	},
	"F07": {
		"AVALANCHE": 0,
		"RANDOM_2.1": 0,
		"RANDOM_2.3": 0,
		"RANDOM_2.13": 0				
	}
}

In [3]:
def avalanche_effect(cipher: SPN80 | Feistel80, verbose = False) -> float:
	messages = [(1 << i) - 1 for i in range(1, 81)]

	prev_enc = cipher.encrypt_block(0)
	total_bits_diff = 0

	for index, data in enumerate(messages, start=1):
		ciphertext = cipher.encrypt_block(data)
		diff = ciphertext ^ prev_enc
		total_bits_diff += diff.bit_count()
		prev_enc = ciphertext

		if verbose:
			print()
			print(f"message {index}")
			print(f"  {'hex':11s}  0x{data:0{BLOCK_BYTES * 2}x}")
			print(f"  {'enc':11s}  0x{ciphertext:0{BLOCK_BYTES * 2}x}")
			print(f"  {'dif':11s}  0x{diff:0{BLOCK_BYTES * 2}x}")

	return total_bits_diff/len(messages)

In [4]:
print("Avalanche Effect Analysis")
for key, design in CIPHERS.items():
	print(key)

	mode = ('SPN' if key[0] == 'S' else 'Feistel')
	cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

	print(f"{'cipher:':8s}{mode}, {cipher.rounds} rounds")

	avg_bits_flipped = avalanche_effect(cipher)

	ANALYSIS[key]["AVALANCHE"] = avg_bits_flipped

	print(f"avg diff is {avg_bits_flipped} bit(s) per plaintext bit flipped")
	print(f"avg expected diff is {BLOCK_SIZE // 2} bit(s) per plaintext bit flipped")
	print()


Avalanche Effect Analysis
S07
cipher: SPN, 4 rounds
avg diff is 1.0 bit(s) per plaintext bit flipped
avg expected diff is 40 bit(s) per plaintext bit flipped

F01
cipher: Feistel, 8 rounds
avg diff is 2.4625 bit(s) per plaintext bit flipped
avg expected diff is 40 bit(s) per plaintext bit flipped

F07
cipher: Feistel, 8 rounds
avg diff is 3.5 bit(s) per plaintext bit flipped
avg expected diff is 40 bit(s) per plaintext bit flipped



In [5]:
long_message = "Tudo o que eu percebo no mundo, percebo em mim. Isto nada mais é do que o meu corpo afetado pelo mundo. Se você vai ter que conviver com você mesmo até o fim, se você vai ter que se aguentar até o fim, se você vai ser espectador de você mesmo até o fim, é melhor que se encante com o que faz!"

print(f"0x{long_message.encode().hex()}")

0x5475646f206f20717565206575207065726365626f206e6f206d756e646f2c207065726365626f20656d206d696d2e204973746f206e616461206d61697320c3a920646f20717565206f206d657520636f72706f206166657461646f2070656c6f206d756e646f2e20536520766f63c3aa20766169207465722071756520636f6e766976657220636f6d20766f63c3aa206d65736d6f206174c3a9206f2066696d2c20736520766f63c3aa20766169207465722071756520736520616775656e746172206174c3a9206f2066696d2c20736520766f63c3aa20766169207365722065737065637461646f7220646520766f63c3aa206d65736d6f206174c3a9206f2066696d2c20c3a9206d656c686f722071756520736520656e63616e746520636f6d206f207175652066617a21


In [6]:
def nistFrequencyTest(bits_string: str) -> bool:
    from math import sqrt, erfc

    def abs(x: float) -> float:
        return x if x >= 0 else -x
    
    bits = bits_string[2:]
    n = len(bits)

    s = 0
    for bit in bits:
        if bit == '0':
            s -= 1
        else:
            s += 1

    s_obs = abs(s) / sqrt(n)
    p_value = erfc(s_obs / sqrt(2))
    
    return p_value >= 0.01

In [7]:
print("NIST - 2.1 Frequency (Monobit) Test\n")
for key, design in CIPHERS.items():
	print(f"{key}: ", end='')

	counter = 0
	for _ in range(100):
		mode = ('SPN' if key[0] == 'S' else 'Feistel')
		cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

		message = long_message.encode()
		message_hex = f"0x{message.hex()}"
		nonce = random_nonce()

		ciphertext = apply_ctr(cipher, nonce, message)
		ciphertext_bin = f"0b{(bin(int.from_bytes(ciphertext))[2:]).zfill(2416)}"
		ciphertext_hex = f"0x{ciphertext.hex()}"

		is_random = nistFrequencyTest(ciphertext_bin)

		if not is_random:
			counter += 1

	ANALYSIS[key]["RANDOM_2.1"] = counter
	print(f"The sequence was considered not random {counter} times")

NIST - 2.1 Frequency (Monobit) Test

S07: The sequence was considered not random 22 times
F01: The sequence was considered not random 29 times
F07: The sequence was considered not random 40 times


In [8]:
def nistRunsTest(bits_string: str) -> bool:
	from math import sqrt, erfc

	def abs(x: float) -> float:
		return x if x >= 0 else -x

	bits = bits_string[2:]
	n = len(bits)

	pi = 0
	for bit in bits:
		pi += int(bit)
	pi /= n

	if abs(pi - 0.5) >= (2/sqrt(n)):
		return False

	v_obs = 1
	for i in range(len(bits)-1):
		if bits[i] == bits[i+1]:
			v_obs += 1

	p_value = erfc((abs(v_obs - (2*n*pi*(1-pi))))/((2*sqrt(2*n)*pi*(1-pi))))
	
	return p_value >= 0.01

In [9]:
print("NIST - 2.3 Runs Test\n")
for key, design in CIPHERS.items():
	print(f"{key}: ", end='')
	
	counter = 0
	for _ in range(100):
		mode = ('SPN' if key[0] == 'S' else 'Feistel')
		cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

		message = long_message.encode()
		message_hex = f"0x{message.hex()}"
		nonce = random_nonce()

		ciphertext = apply_ctr(cipher, nonce, message)
		ciphertext_bin = f"0b{(bin(int.from_bytes(ciphertext))[2:]).zfill(2416)}"
		ciphertext_hex = f"0x{ciphertext.hex()}"

		is_random = nistRunsTest(ciphertext_bin)

		if not is_random:
			counter += 1

	ANALYSIS[key]["RANDOM_2.3"] = counter
	print(f"The sequence was considered not random {counter} times")

NIST - 2.3 Runs Test

S07: The sequence was considered not random 26 times
F01: The sequence was considered not random 29 times
F07: The sequence was considered not random 33 times


In [10]:
def nistCusumTest(bits_string: str, mode: 0 | 1 = 0) -> bool:
	from math import sqrt, erfc, floor

	def phi(x: float) -> float:
		return 0.5 * erfc(-x / sqrt(2))

	bits = bits_string[2:]
	n = len(bits)

	X = []
	for bit in bits:
		if bit == '0':
			X.append(-1)
		else:
			X.append(1)

	cusum = X.copy()

	if mode == 1:
		cusum = cusum[::-1]

	for i in range(1, len(X)):
		cusum[i] += cusum[i-1]

	z = max(cusum)

	if z == 0:
		return False

	z_sqrtN = z/sqrt(n)

	p_value = 1
	lw = floor(((-n/z)-3)/4)
	md = floor(((-n/z)+1)/4)
	up = floor(((n/z)-1)/4)

	for k in range(md, up+1):
		p_value -= (phi(((4*k)+1)*z_sqrtN) - phi(((4*k)-1)*z_sqrtN))

	for k in range(lw, up+1):
		p_value += (phi(((4*k)+3)*z_sqrtN) - phi(((4*k)+1)*z_sqrtN))
	
	return p_value >= 0.01

In [11]:
print("NIST - 2.13 Cumulative Sums (Cusum) Test\n")
for key, design in CIPHERS.items():
	print(f"{key}: ", end='')
	
	counter = 0
	for _ in range(100):
		mode = ('SPN' if key[0] == 'S' else 'Feistel')
		cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

		message = long_message.encode()
		message_hex = f"0x{message.hex()}"
		nonce = random_nonce()

		ciphertext = apply_ctr(cipher, nonce, message)
		ciphertext_bin = f"0b{(bin(int.from_bytes(ciphertext))[2:]).zfill(2416)}"
		ciphertext_hex = f"0x{ciphertext.hex()}"

		is_random = nistCusumTest(ciphertext_bin)

		if not is_random:
			counter += 1

	ANALYSIS[key]["RANDOM_2.13"] = counter
	print(f"The sequence was considered not random {counter} times")

NIST - 2.13 Cumulative Sums (Cusum) Test

S07: The sequence was considered not random 22 times
F01: The sequence was considered not random 15 times
F07: The sequence was considered not random 19 times
